# Train and segment one area

This notebook walks through a complete single-area tutorial:

1. prepare one labeled tile
2. build SAM training chips
3. train a SAM decoder
4. inspect SAM proposal outputs
5. build StarDist training records
6. train StarDist
7. run segmentation

This is a tutorial for understanding and exercising the full pipeline.
It is not a recommended evaluation setup.
For real model assessment, keep separate training and held-out areas.

## Inputs

This notebook assumes the clipped rasters are already in the repo `data/` directory.

You still need to provide a crown-label file for training.
That label file must overlap the clipped rasters.

In [ ]:
from pathlib import Path

from samstars.data import build_scale_stats_from_proposal_records, build_single_tile
from samstars.segmentation import run_segmentation
from samstars.training.chips import build_training_chips
from samstars.training.records import build_stardist_training_records
from samstars.training.sam_training import train_decoder
from samstars.training.stardist_training import train_stardist_model

## Set paths

Update the label and checkpoint paths for your own setup.

In [ ]:
data_dir = Path("data")

img = data_dir / "img_clipped.tif"
chm = data_dir / "chm_clipped.tif"
den = data_dir / "den_clipped.tif"
crowns = Path("/abs/path/crowns.gpkg")
sam_base_ckpt = Path("/abs/path/sam_vit_l_0b3195.pth")
work_dir = data_dir / "train_and_segment_one_area_run"
device = "cpu"

labeled_tiles_dir = work_dir / "labeled_tiles"
training_chips_dir = work_dir / "training_chips"
training_run_dir = work_dir / "training_run"
records_dir = work_dir / "stardist_training_records"
models_dir = work_dir / "models"
inference_dir = work_dir / "segmentation_run"
sam_ckpt = models_dir / "sam_crowns_decoder.pth"
stardist_model = models_dir / "stardist"

## Set tutorial parameters

These are explicit on purpose.
Do not treat them as fixed project defaults.
Change them, rerun the notebook, and compare the intermediate proposal outputs before committing to StarDist training.

In [ ]:
chip_size = 512
chip_stride = 256

sam_epochs = 15
stardist_epochs = 50
val_fraction = 0.2

crop_smooth_sigma = 2.0
smooth_sigma = 4.0
post_smooth_sigma = 0.0

stardist_input_key = "index"
use_aux = True

## Prepare one labeled tile

This creates the tile layout used by the downstream training helpers.

In [ ]:
build_single_tile(
    img=img,
    chm=chm,
    den=den,
    crowns=crowns,
    out=labeled_tiles_dir,
    skip_existing=True,
)

## Build SAM training chips

In [ ]:
build_training_chips(
    tiles_dir=labeled_tiles_dir,
    out_dir=training_chips_dir,
    chip_size=chip_size,
    stride=chip_stride,
)

## Train models

This section first trains the SAM decoder, then creates and inspects proposal rasters, and only then trains StarDist.

In [ ]:
train_decoder(
    chip_dir=training_chips_dir,
    ckpt=sam_base_ckpt,
    out_path=sam_ckpt,
    epochs=sam_epochs,
    device=device,
)

In [ ]:
proposal_result = run_segmentation(
    tiles_dir=labeled_tiles_dir,
    out_dir=training_run_dir,
    sam_ckpt=sam_ckpt,
    device=device,
    run_sam=True,
    run_stardist=False,
    crop_smooth_sigma=crop_smooth_sigma,
    smooth_sigma=smooth_sigma,
    post_smooth_sigma=post_smooth_sigma,
)

## Inspect SAM proposal outputs

Inspect the proposal rasters before continuing.
If the proposals are poor, adjust the parameters above and rerun this section before training StarDist.

In [ ]:
proposal_result

In [ ]:
proposal_rasters_dir = proposal_result["proposals_root"] / "0474" / "rasters"
sorted(p.name for p in proposal_rasters_dir.glob("*.tif"))[:20]

## Build StarDist training records

In [ ]:
train_records, val_records = build_stardist_training_records(
    proposal_records=proposal_result["proposal_records"],
    tiles_dir=labeled_tiles_dir,
    out_dir=records_dir,
    val_fraction=val_fraction,
)

In [ ]:
scale_stats = work_dir / "training_scale_stats.json"

build_scale_stats_from_proposal_records(
    proposal_records=proposal_result["proposal_records"],
    out=scale_stats,
    percentiles=(2.0, 98.0),
    chm_key="chm",
    nodata_values=(-9999.0,),
)

In [ ]:
train_stardist_model(
    train_records=train_records,
    val_records=val_records,
    out_dir=stardist_model.parent,
    epochs=stardist_epochs,
    input_key=stardist_input_key,
    use_aux=use_aux,
    scale_stats_path=scale_stats,
)

## Run segmentation

In [ ]:
result = run_segmentation(
    img=img,
    chm=chm,
    den=den,
    out_dir=inference_dir,
    sam_ckpt=sam_ckpt,
    stardist_model=stardist_model,
    stardist_input_key=stardist_input_key,
    device=device,
    crop_smooth_sigma=crop_smooth_sigma,
    smooth_sigma=smooth_sigma,
    post_smooth_sigma=post_smooth_sigma,
)

## Final output

In [ ]:
inference_dir / "crowns" / "crowns.gpkg"